# GlobalMart — Genie Space Governance Setup

Three governance layers for the Genie Space:

1. **Metric Views** — centralised KPI definitions (named `metricview_*` to avoid confusion with existing `mv_*` Materialized Views)
2. **Access Groups + GRANT** — role-based access per team
3. **Row Filters** — row-level security on sensitive tables

**Run order:** Steps 1–5, once each. Requires workspace admin or catalog owner privileges.

---

## Step 1 — Metric Views

**Correct YAML join syntax (from Databricks docs):**
- Source table columns use the `source.` prefix in the `on` condition
- Joined table columns use the join `name` as prefix
- `on` must be quoted as `'on':` — YAML 1.1 parsers treat unquoted `on` as a boolean
- `using:` can be used instead when the column name is identical in both tables

**Query syntax:** `SELECT Region, MEASURE(\`Total Revenue\`) FROM metricview_revenue GROUP BY ALL`

In [0]:
# ── Metric View 1: Revenue & Profit ──────────────────────────────────────
# fact_sales is the source. FK columns: customer_id (shared name), order_date_key != date_key
# customer_id is the same in both tables -> use "using"
# order_date_key != date_key -> must use "on" with source. prefix

spark.sql("""
CREATE OR REPLACE VIEW globalmart.gold.metricview_revenue
WITH METRICS LANGUAGE YAML AS $$
version: 1.1
comment: "Revenue and profit KPIs. Source: fact_sales. Use MEASURE() to query measures."
source: globalmart.gold.fact_sales
joins:
  - name: customers
    source: globalmart.gold.dim_customers
    using:
      - customer_id
  - name: dates
    source: globalmart.gold.dim_dates
    'on': source.order_date_key = dates.date_key
dimensions:
  - name: Region
    expr: customers.region
    comment: "Customer region: East, West, Central, South, North"
  - name: Segment
    expr: customers.segment
    comment: "Customer segment: Consumer, Corporate, Home Office"
  - name: Year
    expr: dates.year
  - name: Quarter
    expr: dates.quarter
  - name: Month
    expr: dates.month_name
  - name: Payment Type
    expr: payment_type
measures:
  - name: Total Revenue
    expr: SUM(sales)
    comment: "Sum of all transaction sales amounts"
  - name: Total Profit
    expr: SUM(profit)
    comment: "Sum of net profit across all transactions"
  - name: Profit Margin Pct
    expr: ROUND(SUM(profit) / NULLIF(SUM(sales), 0) * 100, 1)
    comment: "Net profit as a percentage of revenue"
  - name: Order Count
    expr: COUNT(DISTINCT order_id)
    comment: "Number of distinct orders"
  - name: Units Sold
    expr: SUM(quantity)
    comment: "Total units sold"
  - name: Avg Order Value
    expr: ROUND(SUM(sales) / NULLIF(COUNT(DISTINCT order_id), 0), 2)
    comment: "Average revenue per order"
$$
""")
print("metricview_revenue created")


In [0]:
# ── Metric View 2: Vendor Return Rate ────────────────────────────────────
# fact_returns is the source. vendor_id is the same in both tables -> use "using"

spark.sql("""
CREATE OR REPLACE VIEW globalmart.gold.metricview_vendor_returns
WITH METRICS LANGUAGE YAML AS $$
version: 1.1
comment: "Vendor return rate KPIs. Source: fact_returns. return_rate_pct is 0-100 scale."
source: globalmart.gold.fact_returns
joins:
  - name: vendors
    source: globalmart.gold.dim_vendors
    using:
      - vendor_id
dimensions:
  - name: Vendor
    expr: vendors.vendor_name
    comment: "Vendor name"
  - name: Return Reason
    expr: return_reason
  - name: Return Status
    expr: return_status
    comment: "approved, pending, or rejected"
  - name: Return Year
    expr: YEAR(return_date)
measures:
  - name: Total Returns
    expr: COUNT(return_key)
    comment: "Number of return transactions"
  - name: Total Refund Amount
    expr: ROUND(SUM(refund_amount), 2)
    comment: "Total dollar value refunded"
  - name: Avg Refund Amount
    expr: ROUND(AVG(refund_amount), 2)
    comment: "Average refund per return"
  - name: Return Rate Pct
    expr: ROUND(COUNT(return_key) * 100.0 / NULLIF(COUNT(DISTINCT order_id), 0), 1)
    comment: "Returns as a percentage of orders. Above 20 is high."
$$
""")
print("metricview_vendor_returns created")


In [0]:
# ── Metric View 3: Slow-Moving Inventory ─────────────────────────────────
# mv_slow_moving_products is the source. product_id is the same in both tables -> use "using"

spark.sql("""
CREATE OR REPLACE VIEW globalmart.gold.metricview_inventory
WITH METRICS LANGUAGE YAML AS $$
version: 1.1
comment: "Inventory velocity KPIs. Source: mv_slow_moving_products. Above 90 days = slow-moving."
source: globalmart.gold.mv_slow_moving_products
joins:
  - name: products
    source: globalmart.gold.dim_products
    using:
      - product_id
dimensions:
  - name: Region
    expr: region
    comment: "Region where product velocity is measured"
  - name: Brand
    expr: products.brand
  - name: Category
    expr: products.categories
  - name: Slow Moving Flag
    expr: CASE WHEN days_since_last_sale > 90 THEN 'Yes' ELSE 'No' END
    comment: "Yes = not sold in over 90 days in this region"
measures:
  - name: Slow Product Count
    expr: COUNT(CASE WHEN days_since_last_sale > 90 THEN 1 END)
    comment: "Number of products with no sales in over 90 days"
  - name: Avg Days Since Last Sale
    expr: ROUND(AVG(days_since_last_sale), 0)
    comment: "Average days since last sale"
  - name: Total Quantity Sold
    expr: SUM(total_quantity_sold)
  - name: Total Sales
    expr: ROUND(SUM(total_sales), 2)
$$
""")
print("metricview_inventory created")

spark.sql("USE CATALOG globalmart")
print("\nAll metric views:")
display(spark.sql("SHOW VIEWS IN globalmart.gold").filter("viewName LIKE 'metricview_%'"))

In [0]:
# ── Sample queries to verify all three metric views ──────────────────────

print("Revenue by Region:")
display(spark.sql("""
    SELECT `Region`,
           MEASURE(`Total Revenue`),
           MEASURE(`Total Profit`),
           MEASURE(`Profit Margin Pct`)
    FROM globalmart.gold.metricview_revenue
    GROUP BY ALL
    ORDER BY MEASURE(`Total Revenue`) DESC
"""))

print("\nVendor Return Rates:")
display(spark.sql("""
    SELECT `Vendor`,
           MEASURE(`Total Returns`),
           MEASURE(`Return Rate Pct`),
           MEASURE(`Total Refund Amount`)
    FROM globalmart.gold.metricview_vendor_returns
    GROUP BY ALL
    ORDER BY MEASURE(`Return Rate Pct`) DESC
"""))

print("\nSlow-Moving by Region:")
display(spark.sql("""
    SELECT `Region`,
           MEASURE(`Slow Product Count`),
           MEASURE(`Avg Days Since Last Sale`)
    FROM globalmart.gold.metricview_inventory
    GROUP BY ALL
    ORDER BY MEASURE(`Slow Product Count`) DESC
"""))


## Step 2 — Access Groups and GRANT permissions

**Create these three groups in the UI first:**
Settings → Identity & Access → Groups → Add group

| Group | Who | Tables |
|---|---|---|
| `globalmart_merchandising` | Merchandising analysts | Products, vendors, inventory, RAG history |
| `globalmart_finance` | Finance and audit | Sales, orders, revenue, DQ audit |
| `globalmart_returns` | Returns investigators | Returns, customer history, fraud scores |

In [0]:
# ── Create missing groups if not present ──────────────────────────────────
groups = ["globalmart_merchandising", "globalmart_finance", "globalmart_returns"]
existing_groups = [
    row['name'] for row in spark.sql("SHOW GROUPS").collect()
]
for group in groups:
    if group not in existing_groups:
        try:
            spark.sql(f"CREATE GROUP `{group}`")
            print(f"Group '{group}' created.")
        except Exception as e:
            print(f"ERROR: Could not create group '{group}': {e}")

# Refresh group list after attempting creation
existing_groups = [
    row['name'] for row in spark.sql("SHOW GROUPS").collect()
]

# ── Shared: all groups need catalog and schema access ─────────────────────
for group in groups:
    if group in existing_groups:
        try:
            spark.sql(f"GRANT USE CATALOG ON CATALOG globalmart TO `{group}`")
            spark.sql(f"GRANT USE SCHEMA ON SCHEMA globalmart.gold TO `{group}`")
        except Exception as e:
            print(f"ERROR: Could not grant catalog/schema access to '{group}': {e}")
    else:
        print(f"WARNING: Group '{group}' does not exist. Skipping grants.")
print("Catalog and schema access granted to all existing groups")

# ── Merchandising ─────────────────────────────────────────────────────────
merchandising_tables = [
    "dim_products", "dim_vendors",
    "mv_slow_moving_products", "mv_return_rate_by_vendor",
    "metricview_inventory", "metricview_vendor_returns",
    "rag_query_history",
]
if "globalmart_merchandising" in existing_groups:
    for tbl in merchandising_tables:
        try:
            spark.sql(f"GRANT SELECT ON TABLE globalmart.gold.{tbl} TO `globalmart_merchandising`")
        except Exception as e:
            print(f"ERROR: Could not grant SELECT on {tbl} to globalmart_merchandising: {e}")
    print(f"Merchandising grants done: {merchandising_tables}")
else:
    print("WARNING: Group 'globalmart_merchandising' does not exist. Merchandising grants skipped.")

# ── Finance ───────────────────────────────────────────────────────────────
finance_tables = [
    "fact_sales", "fact_orders", "dim_customers", "dim_dates",
    "mv_revenue_by_region", "metricview_revenue",
    "dq_audit_report", "pipeline_run_log",
]
if "globalmart_finance" in existing_groups:
    for tbl in finance_tables:
        try:
            spark.sql(f"GRANT SELECT ON TABLE globalmart.gold.{tbl} TO `globalmart_finance`")
        except Exception as e:
            print(f"ERROR: Could not grant SELECT on {tbl} to globalmart_finance: {e}")
    print(f"Finance grants done: {finance_tables}")
else:
    print("WARNING: Group 'globalmart_finance' does not exist. Finance grants skipped.")

# ── Returns team ──────────────────────────────────────────────────────────
returns_tables = [
    "fact_returns", "dim_customers",
    "mv_customer_return_history", "flagged_return_customers",
]
if "globalmart_returns" in existing_groups:
    for tbl in returns_tables:
        try:
            spark.sql(f"GRANT SELECT ON TABLE globalmart.gold.{tbl} TO `globalmart_returns`")
        except Exception as e:
            print(f"ERROR: Could not grant SELECT on {tbl} to globalmart_returns: {e}")
    print(f"Returns grants done: {returns_tables}")
else:
    print("WARNING: Group 'globalmart_returns' does not exist. Returns grants skipped.")

print("\nGrant verification on fact_sales:")
display(spark.sql("SHOW GRANTS ON TABLE globalmart.gold.fact_sales"))

## Step 3 — Row Filters

Row filters are SQL UDFs that fire automatically at query time. Original data is unchanged — each user just sees the rows they are permitted to see.

1. **Region filter on `dim_customers`** — controls which customer regions each group sees
2. **Return status filter on `fact_returns`** — returns team sees approved/pending only; finance sees all including rejected

In [0]:
# ── Row Filter 1: dim_customers — region access ───────────────────────────
spark.sql("""
CREATE OR REPLACE FUNCTION globalmart.gold.filter_customers_by_region(region STRING)
RETURNS BOOLEAN
RETURN
    CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('admins')                   THEN TRUE
        WHEN IS_ACCOUNT_GROUP_MEMBER('globalmart_finance')       THEN TRUE
        WHEN IS_ACCOUNT_GROUP_MEMBER('globalmart_returns')       THEN TRUE
        WHEN IS_ACCOUNT_GROUP_MEMBER('globalmart_merchandising') THEN TRUE
        ELSE FALSE
    END
""")
spark.sql("""
ALTER TABLE globalmart.gold.dim_customers
SET ROW FILTER globalmart.gold.filter_customers_by_region ON (region)
""")
print("Row filter applied to dim_customers")


# ── Row Filter 2: fact_returns — status access ────────────────────────────
spark.sql("""
CREATE OR REPLACE FUNCTION globalmart.gold.filter_returns_by_status(return_status STRING)
RETURNS BOOLEAN
RETURN
    CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('admins')             THEN TRUE
        WHEN IS_ACCOUNT_GROUP_MEMBER('globalmart_finance') THEN TRUE
        WHEN IS_ACCOUNT_GROUP_MEMBER('globalmart_returns') THEN
            return_status IN ('approved', 'pending', 'Approved', 'Pending')
        ELSE FALSE
    END
""")
spark.sql("""
ALTER TABLE globalmart.gold.fact_returns
SET ROW FILTER globalmart.gold.filter_returns_by_status ON (return_status)
""")
print("Row filter applied to fact_returns")


## Step 4 — Grant EXECUTE on row filter functions

All groups need EXECUTE on the UDF functions to query tables that have row filters applied.

In [0]:
udfs = [
    "globalmart.gold.filter_customers_by_region",
    "globalmart.gold.filter_returns_by_status",
]
for group in ["globalmart_merchandising", "globalmart_finance", "globalmart_returns"]:
    for udf in udfs:
        spark.sql(f"GRANT EXECUTE ON FUNCTION {udf} TO `{group}`")
print("EXECUTE grants done for all row filter functions")


## Step 5 — Verify

In [0]:
print("Metric views (metricview_*):")
display(spark.sql("SHOW VIEWS IN globalmart.gold").filter("viewName LIKE 'metricview_%'"))

print("\nRow filters registered:")
display(spark.sql("""
    SELECT table_name, filter_name
    FROM globalmart.information_schema.row_filters
    WHERE table_schema = 'gold'
"""))

print("\nGrants on dim_customers:")
display(spark.sql("SHOW GRANTS ON TABLE globalmart.gold.dim_customers"))


## Step 6 — Add metric views to the Genie Space

Configure → Data → Add:
- `globalmart.gold.metricview_revenue`
- `globalmart.gold.metricview_vendor_returns`
- `globalmart.gold.metricview_inventory`

Row filters are transparent to Genie — they fire at the Unity Catalog layer automatically.

Share → Add each group → CAN RUN:
- `globalmart_merchandising`
- `globalmart_finance`
- `globalmart_returns`